In [ ]:
DEBUG = True

In [ ]:
import sys
import torch
import matplotlib.pyplot as plt

In [ ]:
sys.executable

In [ ]:
sys.path

In [ ]:
from pfs.ga.bayesian import Model
from pfs.ga.bayesian.distributions import Normal, Uniform, Dirichlet, Categorical
from pfs.ga.bayesian.proposals import NormalProposal, DirichletProposal, CategoricalProposal
from pfs.ga.bayesian import MCMC
from pfs.ga.bayesian.kernels import GibbsKernel

In [ ]:
if DEBUG and 'debugpy' not in globals():
    import debugpy
    debugpy.listen(5678)
    print("Debugger is listening on port 5678. You can attach to it now.")

In [ ]:
class Mixture(Model):

    def __init__(self, *args, N=(1000,), **kwargs):
        super().__init__(*args, **kwargs)

        self.N = N

    def model(self, context):
        K = 2
        N = self.N

        w = context.sample('w', Dirichlet(context.tensor(1.0 * torch.ones(K))))
        theta_1 = context.sample('theta_1', Uniform(context.tensor(-1.0), context.tensor(0.0), validate_args=False))
        theta_2 = context.sample('theta_2', Uniform(context.tensor(2.0), context.tensor(3.0), validate_args=False))

        with context.plate("n", N):
            z = context.sample('z', Categorical(w))
            x_1 = context.sample('x_1', Normal(theta_1, context.tensor(1.0)))
            x_2 = context.sample('x_2', Normal(theta_2, context.tensor(1.5)))
            x = context.select('x', [x_1, x_2], z)
            obs = context.sample('obs', Normal(x, context.tensor(0.25)), observed=True)

    def step(self, context):
        # Define the Gibbs blocks for each group of sampled variables
        context.step(
            'w',
            [ self.w ],
            proposal = DirichletProposal(3.0 * torch.ones_like(self.w.value(context.state)))
        )
        
        context.step(
            'theta_1',
            [ self.theta_1 ],
            proposal = NormalProposal(self.theta_1.value(context.state), context.tensor(0.5))
        )
        
        context.step(
            'theta_2',
            [ self.theta_2 ],
            proposal = NormalProposal(self.theta_2.value(context.state), context.tensor(0.5))
        )

        context.step(
            'z',
            [ self.z ],
            proposal = CategoricalProposal(self.w.value(context.state).expand(self.N + self.w.value(context.state).shape))
        )

        context.step(
            'x_1',
            [ self.x_1 ],
            proposal = NormalProposal(self.x_1.value(context.state), context.tensor(1.0))
        )
        
        context.step(
            'x_2',
            [ self.x_2 ],
            proposal = NormalProposal(self.x_2.value(context.state), context.tensor(3.0))
        )

In [ ]:
model = Mixture(device='cpu')
model.build()

init_state = model.sample()
observed = { 'obs': init_state['obs'].clone() }

In [ ]:
init_state['w'].device

In [ ]:
# Print the population weights
print('w', model.w.value(init_state))

In [ ]:
# Print the hyperparameters
print('theta_1', model.theta_1.value(init_state))
print('theta_2', model.theta_2.value(init_state))

In [ ]:
# Plot the distribution of observed data
hist, bins = torch.histogram(observed['obs'].cpu(), bins=30)
plt.step(bins[:-1], hist, where='post')

plt.axvline(model.theta_1.value(init_state).cpu(), color='red', linestyle='--', label='True theta 1')
plt.axvline(model.theta_2.value(init_state).cpu(), color='blue', linestyle='--', label='True theta 2')

# Plot the components of the mixture
hist, bins = torch.histogram(observed['obs'][init_state['z'] == 0].cpu(), bins=30)
plt.step(bins[:-1], hist, where='post', color='red', linestyle='--', label='Component 1')

hist, bins = torch.histogram(observed['obs'][init_state['z'] == 1].cpu(), bins=30)
plt.step(bins[:-1], hist, where='post', color='blue', linestyle='--', label='Component 2')

In [ ]:
# Plot the distribution of observed data
hist, bins = torch.histogram(observed['obs'].cpu(), bins=30, density=True)
plt.step(bins[:-1], hist, where='post')

plt.axvline(model.theta_1.value(init_state).cpu(), color='red', linestyle='--', label='True theta 1')
plt.axvline(model.theta_2.value(init_state).cpu(), color='blue', linestyle='--', label='True theta 2')

# Plot the components of the mixture
hist, bins = torch.histogram(observed['obs'][init_state['z'] == 0].cpu(), bins=30, density=True)
plt.step(bins[:-1], hist, where='post', color='red', linestyle='--', label='Component 1')

hist, bins = torch.histogram(observed['obs'][init_state['z'] == 1].cpu(), bins=30, density=True)
plt.step(bins[:-1], hist, where='post', color='blue', linestyle='--', label='Component 2')

# Plot the underlying distribution of the data
xx = torch.linspace(-5, 10, 100)
pdf_0 = Normal(init_state['theta_1'].cpu(), 1.0).log_prob(xx).exp() * model.w.value(init_state)[0].cpu()
pdf_1 = Normal(init_state['theta_2'].cpu(), 3.0).log_prob(xx).exp() * model.w.value(init_state)[1].cpu()
plt.plot(xx, pdf_0, color='red', linestyle='--', label='Component 1 PDF')
plt.plot(xx, pdf_1, color='blue', linestyle='--', label='Component 2 PDF')
plt.plot(xx, pdf_0 + pdf_1, color='black', linestyle='--', label='Mixture PDF')

In [ ]:
kernel = GibbsKernel(model)
mcmc = MCMC(kernel,
            num_warmup=10000, num_samples=10000, num_chains=10,
            thinning=100,
            progress=True)

In [ ]:
for k in init_state:
    print(k, init_state[k].device)

In [ ]:
for k in observed:
    print(k, observed[k].device)

In [ ]:
mcmc.run(observed=observed)

In [ ]:
mcmc.trace['theta_1'].shape

In [ ]:
print('theta_1', init_state['theta_1'])
print('theta_1', torch.mean(mcmc.trace['theta_1'], dim=0))
print('theta_2', init_state['theta_2'])
print('theta_2', torch.mean(mcmc.trace['theta_2'], dim=0))

In [ ]:
for i in range(mcmc.trace['theta_1'].shape[-1]):
    hist, bins = torch.histogram(mcmc.trace['theta_1'][:, i].flatten(), bins=30, density=True)
    plt.step(bins[:-1], hist, where='post')

plt.axvline(init_state['theta_1'].cpu(), color='red', linestyle='--', label='True theta_1')

In [ ]:
for i in range(mcmc.trace['theta_2'].shape[-1]):
    hist, bins = torch.histogram(mcmc.trace['theta_2'][:, i].flatten(), bins=30, density=True)
    plt.step(bins[:-1], hist, where='post')

plt.axvline(init_state['theta_2'], color='red', linestyle='--', label='True theta_2')

In [ ]:
for i in range(mcmc.trace['theta_1'].shape[-1]):
    plt.plot(mcmc.trace['theta_1'][..., i], '.')

plt.axhline(init_state['theta_1'].cpu(), color='red', linestyle='--', label='True theta')

In [ ]:
for i in range(mcmc.trace['theta_2'].shape[-1]):
    plt.plot(mcmc.trace['theta_2'][..., i], '.')

plt.axhline(init_state['theta_2'].cpu(), color='red', linestyle='--', label='True theta_2')

In [ ]:
mcmc.trace['x'].shape, observed['obs'].shape

In [ ]:
k = 5
for i in range(mcmc.trace['x'].shape[-1]):
    plt.plot(mcmc.trace['x'][:, k, i], '.')

plt.axhline(observed['obs'][k].cpu(), color='red', linestyle='--', label='True theta')

In [ ]:
init_state['x'].shape, mcmc.trace['x'].shape

In [ ]:
init_state['x'][:10]

In [ ]:
torch.mean(mcmc.trace['x'], dim=(0, -1))[:10]

In [ ]:
# Plot the distribution of observed data
hist, bins = torch.histogram(observed['obs'].cpu(), bins=30, density=True)
plt.step(bins[:-1], hist, where='post')

plt.axvline(model.theta_1.value(init_state).cpu(), color='red', linestyle='--', label='True theta 1')
plt.axvline(model.theta_2.value(init_state).cpu(), color='blue', linestyle='--', label='True theta 2')

plt.axvline(mcmc.trace['theta_1'].mean().cpu(), color='red', linestyle='-', label='Fitted theta 1')
plt.axvline(mcmc.trace['theta_2'].mean().cpu(), color='blue', linestyle='-', label='Fitted theta 2')

# Plot the underlying distribution of the data
xx = torch.linspace(-5, 10, 100)
pdf_0 = Normal(mcmc.trace['theta_1'].mean().cpu(), 1.0).log_prob(xx).exp() * mcmc.trace['w'][..., 0].mean().cpu()
pdf_1 = Normal(mcmc.trace['theta_2'].mean().cpu(), 3.0).log_prob(xx).exp() * mcmc.trace['w'][..., 1].mean().cpu()
plt.plot(xx, pdf_0, color='red', linestyle='--', label='Component 1 PDF')
plt.plot(xx, pdf_1, color='blue', linestyle='--', label='Component 2 PDF')
plt.plot(xx, pdf_0 + pdf_1, color='black', linestyle='--', label='Mixture PDF')

In [ ]:
mcmc.trace['z'].shape, observed['obs'].shape

In [ ]:
# Plot the membership probabilities for each data point as a function of the observed value

p = (mcmc.trace['z'] == 0).sum(dim=(0, -1)) / mcmc.trace['z'].shape[1]
plt.scatter(observed['obs'].cpu(), p.cpu(), s=1)

p = (mcmc.trace['z'] == 1).sum(dim=(0, -1)) / mcmc.trace['z'].shape[1]
plt.scatter(observed['obs'].cpu(), p.cpu(), s=1)

plt.grid()